In [1]:
import numpy as np
from pyscf import gto, scf, cc

####  test H2 monomers ####
a = 2 # bond length in a cluster
d = 4 # distance between each cluster
unit = 'b' # unit of length
na = 2 # size of a cluster (monomer)
nc = 1 # set as integer multiple of monomers
spin = 0 # spin per monomer
frozen = 0 # frozen orbital per monomer
elmt = 'H'
unit = 'B'
basis = 'sto6g'
atoms = ""
for n in range(nc*na):
    shift = ((n - n % na) // na) * (d-a)
    atoms += f"{elmt} {n*a+shift:.5f} 0.00000 0.00000 \n"
###########################

mol = gto.M(atom=atoms,
            basis="sto6g",
            verbose=4,
            unit=unit,
            symmetry=0,
            charge=0,
            spin=spin*nc,
            max_memory=40000,
            )

mf = scf.RHF(mol)
mf.kernel()

mycc = cc.CCSD(mf).set_frozen()
mycc.kernel()

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-29-generic', version='#29~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Mon May 11 10:30:58 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Mon Jun  1 19:24:36 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 2
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry 0 subgroup None
[INPUT] Mole.unit = B
[INPUT] Symbol           X                Y                Z      unit

(np.float64(-0.039641400884833235),
 array([[-5.66005116e-16]]),
 array([[[[-0.19736585]]]]))

In [2]:
options = {'eql_time': 10,
            'n_blocks': 100,
            'n_walkers': 300,
            'max_error': 0.0,
            'mix_precision': False,
            'seed': 17,
            'walker_type': 'rhf',
            'trial': 'pt2ccsd',
            }

from afqmc import integral, launch_afqmc
integral.prep_integral(mycc, chol_cut=1e-5)


Preparing AFQMC calculation
Calculating Cholesky integrals
Finished calculating Cholesky integrals
Size of the correlation space:
Number of electrons:        [1, 1]
Number of basis functions:  2
Number of Cholesky vectors: 3


In [4]:
from jax import lax, vmap, jit, random
from jax import numpy as jnp
from afqmc.sampling import sampler
from dataclasses import dataclass
from functools import partial

@dataclass
class sampler_exp(sampler):

    @partial(jit, static_argnums=(0,1,2))
    def block_sample(
        self,
        prop,
        trial,
        prop_data,
        ham_data,
        wave_data,
        ):
        """Block scan function. Propagation and energy calculation."""
        prop_data["key"], subkey = random.split(prop_data["key"])
        fields = random.normal(
            subkey,
            shape=(
                self.n_prop_steps,
                prop.n_walkers,
                trial.nchol,
            ),
        )
        _step_scan_wrapper = lambda x, y: self._step_scan(
            x, y, ham_data, prop, trial, wave_data
        )
        prop_data, _ = lax.scan(_step_scan_wrapper, prop_data, fields)
        prop_data = prop.orthonormalize_walkers(prop_data)
        prop_data["n_killed_walkers"] = prop_data["weights"].size - jnp.count_nonzero(prop_data["weights"])

        energies = jnp.real(trial.calc_energy(prop_data["walkers"], ham_data, wave_data))
        outlier = jnp.abs(energies - prop_data["e_estimate"]) > jnp.sqrt(2.0 / prop.dt) # 20 Ha for dt = 0.005
        weights = jnp.where(outlier, 0.0, prop_data["weights"])

        guide_olps = trial.calc_overlap(prop_data["walkers"], wave_data)
        trial_olps = trial.calc_trial_overlap(prop_data["walkers"], wave_data)
        prop_data["overlaps"] = guide_olps

        olp_ratio = trial_olps / guide_olps
        weights_p = weights * olp_ratio

        blk_wt = jnp.sum(weights)
        blk_wp = jnp.sum(weights_p)
        blk_et = jnp.sum(weights_p * energies) / blk_wp

        # prop_data["pop_control_ene_shift"] = 0.9 * prop_data["pop_control_ene_shift"] + 0.1 * blk_eg
        prop_data = prop.stochastic_reconfiguration_local(prop_data)
        prop_data["overlaps"] = trial.calc_overlap(prop_data["walkers"], wave_data)

        return prop_data, (blk_wt, blk_wp, blk_et)
    
    def __hash__(self) -> int:
        return hash(tuple(self.__dict__.values()))

In [5]:
import h5py
from jax import numpy as jnp
from afqmc import cholesky, slater, propagation
from afqmc.wavefunctions import rwfn_exp

def load_afqmc(options=None,
               amp_file="amplitudes.npz",
               chol_file="FCIDUMP_chol"):

    options["dt"] = options.get("dt", 0.005)
    options["eql_time"] = options.get("eql_time", 20)
    options["n_walkers"] = options.get("n_walkers", 50)
    options["n_prop_steps"] = options.get("n_prop_steps", 50)
    options["n_blocks"] = options.get("n_blocks", 500)
    options["seed"] = options.get("seed", np.random.randint(1, int(1e6)))
    options["n_exp_terms"] = options.get("n_exp_terms",6)
    options["walker_type"] = options.get("walker_type", "rhf")
    options["trial"] = options.get("trial", None)
    options["free_projection"] = options.get("free_projection", False)
    options["n_batch"] = options.get("n_batch", 1)
    options["max_error"] = options.get("max_error", 0.0)
    options["nchol_chunk"] = options.get("nchol_chunk", 100)
    options["max_memory"] = options.get("max_memory", 2000) # MB
    options["mix_precision"] = options.get("mix_precision", True)

    print("\nLoad system from Integral File")

    with h5py.File(chol_file, "r") as fh5:
        [nelec, norb, ms] = fh5["header"]
        spin_type = fh5["spin_type"][()]
        h0 = jnp.array(fh5.get("energy_core"))
        h1 = jnp.array(fh5.get("hcore"))
        chol = jnp.array(fh5.get("chol"))
        h1_mod = jnp.array(fh5.get("hcore_mod"))
    
    if isinstance(spin_type, bytes):
        spin_type = spin_type.decode()

    assert spin_type in ["restricted", "unrestricted"]

    if spin_type == 'restricted':
        h1 = jnp.array(h1).reshape(norb, norb)
        h1_mod = jnp.array(h1_mod).reshape(norb, norb)
        chol = jnp.array(chol).reshape(-1, norb, norb)

    elif spin_type == 'unrestricted':
        h1 = jnp.array(h1).reshape(2, norb, norb)
        h1_mod = jnp.array(h1_mod).reshape(2, norb, norb)
        chol = jnp.array(chol).reshape(2, -1, norb, norb)

    assert type(ms) is np.int64
    assert type(nelec) is np.int64
    assert type(norb) is np.int64

    ms, nelec, norb = int(ms), int(nelec), int(norb)
    nelec_sp = ((nelec + abs(ms)) // 2, (nelec - abs(ms)) // 2)

    ham_data = {}
    ham_data["h0"] = h0

    if spin_type == 'restricted':
        ham_data["h1"] = jnp.array([h1, h1])
        ham_data["h1_mod"] = jnp.array(h1_mod)
        nchol = chol.shape[0]
        ham_data["chol"] = jnp.array(chol.reshape(chol.shape[0], -1))
    elif spin_type == 'unrestricted':
        ham_data["h1"] = jnp.array(h1)
        ham_data["h1_mod"] = jnp.array(h1_mod)
        nchol = chol[0].shape[0]
        ham_data["chol"] = jnp.array([chol[0].reshape(chol[0].shape[0], -1),
                                      chol[1].reshape(chol[1].shape[0], -1)])
        # ham_data["chol"] = [jnp.array(chol[0]), jnp.array(chol[1])]

    options["nchol_chunk"] = cholesky.chunk_chol(
        chol, options["nchol_chunk"], options["max_memory"]/options["n_walkers"])

    wave_data = {}
    mo_coeff = jnp.array([np.eye(norb),np.eye(norb)])

    if spin_type == "restricted":
        nocc = nelec_sp[0]
        wave_data["mo_coeff"] = mo_coeff[0][:,:nocc]
        wave_data["nelec"] = nelec_sp
        wave_data["norb"] = norb
        wave_data["nchol"] = nchol
        wave_data["rdm1"] = jnp.array([wave_data["mo_coeff"] @ wave_data["mo_coeff"].T] * 2)
        
    print("\nQMC System")
    print(f"Number of electrons: {nelec_sp}")
    print(f"Spin Multiplicity:   {ms}")
    print(f"Number of orbitals:  {norb}")
    print(f"Number of Chol:      {nchol}")

    print("\nQMC Parameters")
    for op in options:
        if options[op] is not None:
            print(f"{str(op):<20s}: {str(options[op]):>20s}")

    return ham_data, wave_data, options

In [6]:
options = {'eql_time': 10,
            'n_blocks': 100,
            'n_walkers': 300,
            'max_error': 0.0,
            'mix_precision': False,
            'seed': 17,
            'walker_type': 'rhf',
            'trial': 'pt2ccsd',
            }

ham_data, wave_data, options = load_afqmc(options)


Load system from Integral File
Maximum memory per walker:            6.67 MB
Maximum number of Cholesky per chunk: 109226
Number of Cholesky chunks:            1
Number of Cholesky per chunk:         3
Number of padding Cholesky:           0

QMC System
Number of electrons: (1, 1)
Spin Multiplicity:   0
Number of orbitals:  2
Number of Chol:      3

QMC Parameters
eql_time            :                   10
n_blocks            :                  100
n_walkers           :                  300
max_error           :                  0.0
mix_precision       :                False
seed                :                   17
walker_type         :                  rhf
trial               :              pt2ccsd
dt                  :                0.005
n_prop_steps        :                   50
n_exp_terms         :                    6
free_projection     :                False
n_batch             :                    1
nchol_chunk         :                    3
max_memory          :         

In [7]:
def rhf_overlap(trial, walker, wave_data):
    return slater.r_overlap(wave_data["mo_coeff"], walker)

def rhf_calc_force_bias(trial, walker, ham_data, wave_data):
    chol = ham_data["chol"].reshape(trial.nchol, trial.norb, trial.norb)
    return slater.r_force_bias(wave_data["mo_coeff"], walker, chol)

def rhf_calc_energy(trial, walker, ham_data, wave_data):
    h0 = ham_data["h0"]
    h1 = ((ham_data["h1"][0] + ham_data["h1"][0].T) / 2.0)
    chol = ham_data["chol"].reshape(trial.nchol, trial.norb, trial.norb)
    return slater.r_energy(wave_data["mo_coeff"], walker, h0, h1, chol)

In [8]:
def init_prop_data(
    trial,
    wave_data: dict,
    ham_data: dict,
    options: dict,
) -> dict:
    prop_data = {}
    prop_data["weights"] = jnp.ones(options["n_walkers"])
    prop_data["walkers"] = jnp.array([wave_data["mo_coeff"]]*options["n_walkers"], dtype=jnp.complex128)
    energies = jnp.real(trial.calc_energy(prop_data["walkers"], ham_data, wave_data))
    e_estimate = jnp.array(jnp.sum(energies) / options["n_walkers"])
    prop_data["e_estimate"] = e_estimate
    prop_data["pop_control_ene_shift"] = e_estimate
    prop_data["overlaps"] = trial.calc_overlap(prop_data["walkers"], wave_data)
    prop_data["key"] = random.PRNGKey(options["seed"])
    prop_data["n_killed_walkers"] = 0
    return prop_data

In [9]:
prop = propagation.propagator_restricted(
        options["dt"], 
        options["n_walkers"], 
        options["n_exp_terms"],
        options["n_batch"]
    )

trial = rwfn_exp.rwfn(    
    guide_overlap_fn=rhf_overlap,
    trial_overlap_fn=rhf_overlap,
    force_bias_fn=rhf_calc_force_bias,
    energy_fn=rhf_calc_energy,
    nelec=wave_data["nelec"],
    norb=wave_data["norb"],
    nchol=wave_data["nchol"],
    nchol_chunk=options["nchol_chunk"],
    )

sampler = sampler_exp(
    options["n_prop_steps"],
    options["n_blocks"],
    n_chol=wave_data["nchol"],
    )

In [11]:
import time

import numpy as np

from afqmc import config, prep, sampling

from jax import numpy as jnp
from jax import random

from functools import partial

print = partial(print, flush=True)
init_time = time.time()

prep.print_start()
config.setup_jax()

ham_data = prop._build_propagation_intermediates(ham_data, trial, wave_data)
prop_data = init_prop_data(trial, wave_data, ham_data, options)

init_e = prop_data["e_estimate"]
init_w = np.sum(prop_data["weights"])

print("\nEquilibration")

print(f"{'inv_T':>5s}  "
      f"{'weight':>12s}  {'killW':>5s}  "
      f"{'energy':>12s}  {'runTime':>8s}")

print(f"{0.:5.2f}  "
      f"{init_w:12.6f}  {0:5d}  "
      f"{init_e:12.6f}  {time.time() - init_time:8.2f}")


block_time = prop.dt * sampler.n_prop_steps
neql_block = int(-(-options["eql_time"] // block_time))

for n in range(1, neql_block+1):
    prop_data, (wt, wp, e) \
        = sampler.block_sample(prop, trial, prop_data, ham_data, wave_data)

    if (n+1) % (min(max(neql_block // 10, 1), 20)) == 0 and n > 0:
        nkill = prop_data["n_killed_walkers"]
        print(f"{(n+1)*block_time:5.2f}  "
              f"{wt:12.6f}  {nkill:5d}  "
              f"{e:12.6f}  {time.time() - init_time:8.2f}")


    ________                     _____                    
    ___  __ \___  __________________(_)_____________ _    
    __  /_/ /  / / /_  __ \_  __ \_  /__  __ \_  __ `/    
    _  _, _// /_/ /_  / / /  / / /  / _  / / /  /_/ /     
    /_/ |_| \__,_/ /_/ /_//_/ /_//_/  /_/ /_/_\__, /      
                                             /____/       
    _____________________________  ___________            
    ___    |__  ____/_  __ \__   |/  /_  ____/            
    __  /| |_  /_   _  / / /_  /|_/ /_  /                 
    _  ___ |  __/   / /_/ /_  /  / / / /___               
    /_/  |_/_/      \___\_\/_/  /_/  \____/               



Hostname:     sharmagroup-rn
System:       Linux
Node:         sharmagroup-rn
Release:      6.17.0-29-generic
Machine:      x86_64
Processor:    x86_64
JAX backend:  GPU
JAX devices:  [CudaDevice(id=0)]
Device kind:  NVIDIA GeForce RTX 5060 Ti
Platform:     gpu

Equilibration
inv_T        weight  killW        energy   runTime
 0.00    300.000000      0     -1.056430      1.12
 1.00    300.320862      0  -1.075570+0.000000j      3.35
 2.00    300.573496      0  -1.097327+0.000000j      3.39
 3.00    300.539869      0  -1.096249-0.000000j      3.42
 4.00    300.476808      0  -1.088795+0.000000j      3.46
 5.00    300.548964      0  -1.092692-0.000000j      3.50
 6.00    300.504732      0  -1.084771-0.000000j      3.53
 7.00    300.437130      0  -1.084311+0.000000j      3.57
 8.00    300.517955      0  -1.084974+0.000000j      3.61
 9.00    300.553783      0  -1.095438-0.000000j      3.64
10.00    300.623201      0  -1.111641-0.000000j      3.68
